<small><font color=gray>Notebook author: <a href="https://www.linkedin.com/in/olegmelnikov/" target="_blank">Oleg Melnikov</a>, <a href="https://www.hse.ru/en/staff/sara/" target="_blank">Saraa Ali</a>  ©2026 onwards</font></small><hr style="margin:0;background-color:silver">

**[<font size=6>🧬Genomics</font>](https://www.kaggle.com/t/ade6fdb522504517990bd8b862e50928)**. [**Instructions**](https://colab.research.google.com/drive/1owkYjuRGkx050LQnM3b3yTzd0Dr2XbeV) for running Colabs.

<small>**(Optional) CONSENT.** <mark>[ X ]</mark> We consent to sharing our Colab (after the assignment ends) with other students/instructors for educational purposes. We understand that sharing is optional and this decision will not affect our grade in any way. <font color=gray><i>(If ok with sharing your Colab for educational purposes, leave "X" in the check box.)</i></font></small>

In [1]:
from google.colab import drive; drive.mount('/content/drive')   # OK to enable, if your kaggle.json is stored in Google Drive

Mounted at /content/drive


In [2]:
!pip uninstall -y kaggle kagglesdk kagglehub
!pip install -U kaggle kagglesdk kagglehub

Found existing installation: kaggle 2.0.1
Uninstalling kaggle-2.0.1:
  Successfully uninstalled kaggle-2.0.1
Found existing installation: kagglesdk 0.1.18
Uninstalling kagglesdk-0.1.18:
  Successfully uninstalled kagglesdk-0.1.18
Found existing installation: kagglehub 1.0.0
Uninstalling kagglehub-1.0.0:
  Successfully uninstalled kagglehub-1.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.8/201.8 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 9.1 MB/s eta 0:00:00


In [3]:
!pip -q install --upgrade --force-reinstall --no-deps kaggle > log  # upgrade kaggle package (to avoid a warning)
!mkdir -p ~/.kaggle                               # .kaggle folder must contain kaggle.json for kaggle executable to properly authenticate you to Kaggle.com
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json >log  # First, download kaggle.json from kaggle.com (in Account page) and place it in the root of mounted Google Drive
# !cp kaggle.json ~/.kaggle/kaggle.json > log       # Alternative location of kaggle.json (without a connection to Google Drive)
!chmod 600 ~/.kaggle/kaggle.json                  # give only the owner full read/write access to kaggle.json
!kaggle config set -n competition -v 26-hse-genomics # set the competition context for the next few kaggle API calls. !kaggle config view - shows current settings
!kaggle competitions download >> log              # download competition dataset as a zip file
!unzip -o *.zip >> log                            # Kaggle dataset is copied as a single file and needs to be unzipped.
!kaggle competitions leaderboard --show           # print public leaderboard

- competition is now set to: 26-hse-genomics
100% 9.65M/9.65M [00:00<00:00, 177MB/s]
Using competition: 26-hse-genomics
Next Page Token = CfDJ8CS0IeAoHcJGgSEc27rBbk7X9nPS6cHdWOt_udVtSGRHu2blAV6o6rI_Zci6LQZ2Z7hCq19PTmAHmqif8_XTwAY
  teamId  teamName                       submissionDate              score    
--------  -----------------------------  --------------------------  -------  
15589041  H-hhhh                         2026-04-12 08:42:36.310000  0.98687  
15564267  D-esert Eagle                  2026-04-09 18:18:40.636000  0.98662  
15604189  BF – No, Ain't Got One         2026-04-12 21:27:29.793000  0.98662  
15586776  AK                             2026-04-15 22:28:45.370000  0.98662  
15564353  E-ln(Musk)                     2026-04-09 21:15:07.536000  0.98625  
15565432  BK - Bayerische Motoren Werke  2026-04-14 21:13:00.043000  0.98625  
15589916  T-ee                           2026-04-10 11:14:29.173000  0.98612  
15583964  N-INJERS🥷                      2026-04-13 10:44:1

In [4]:
!nvidia-smi --query-gpu=gpu_name,memory.total,memory.free,memory.used --format=csv # GPU specs

name, memory.total [MiB], memory.free [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 14913 MiB, 0 MiB


In [5]:
# from google.colab import drive
# drive.mount('/content/drive')

In [6]:
%%time
%%capture
%reset -f
!pip -q install -U sentence-transformers > log
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all"
import numpy as np, pandas as pd, time, matplotlib.pyplot as plt, os, plotly
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer as SBERT
from sklearn.svm import SVC, LinearSVC, NuSVC
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='id') # rounds values to 2 decimals

class Timer():
  def __init__(self, lim:'RunTimeLimit'=60*5): self.t0, self.lim, _ = time.time(), lim, print(f'⏳ started. You have {lim} sec. Good luck!')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'\033[91m\033[1m' + msg + f' > {self.lim} sec limit!!!\033[0m' if (time.time()-self.t0-1) > self.lim else msg)

np.set_printoptions(linewidth=10000, precision=4, edgeitems=20, suppress=True)
pd.set_option('display.max_rows', 100, 'display.max_columns', 100, 'display.max_colwidth', 100, 'display.precision', 2, 'display.max_rows', 4)

CPU times: user 16.3 s, sys: 2.11 s, total: 18.4 s
Wall time: 35.3 s


In [7]:
vX = pd.read_csv('testX/testX.csv').set_index('id')
tYX = pd.read_csv('trainYX/trainYX.csv').set_index('id')
vX

,DNA
id,
100000,TTGATTAATAAGATTCCTTGACACCCTTTGTAAAGTTTCTATTTCGTGTGAAATATCTATCTCTTCAAATCCTTTTAATTTATCTAGGTATTTGCT...
100001,ATTAGTAACGGAGGATTTACTAGATGTTTGGATTTATATTCTAATTTTATTCAGGTGGAAGGGATTGTTTTATGATTCAATAGTATACAGAGAATA...
...,...
119998,CGTCGGCATGCTCGGGCAGTGCGGCGGGCCAGCAGCGTGCCAGTTGTCGCGGGGCGGCCGGGCATCGCGGCGCCGGGCGGCAGCACTCCCGCGAAG...
119999,GCGAGGGCACGAAGGCACGACGGCAACGGCGGCGAGGAGCGCTGTGGCAACCGTCTCCGCGTTTGCGTGCGTACAGCCGAGAGCTGGTTCGCGCAG...


In [8]:
tmr = Timer() # runtime limit (in seconds). Add all of your code after the timer

⏳ started. You have 300 sec. Good luck!


❗Do not modify the setup above.

<hr color=red>

<font size=5>⏳</font> <strong><font color=orange size=5>Your Code, Documentation, Ideas and Timer - All Start Here...</font></strong>

Students: Keep all your definitions, code, documentation **between** ⏳ symbols.

## **Task 1. Preprocessing Pipeline**

Explain elements of your preprocessing pipeline i.e. feature engineering, subsampling, clustering, dimensionality reduction, etc.
1. Why did you choose these elements? (Something in EDA, prior experience,...? Btw, EDA is not required)
1. How do you evaluate the effectiveness of these elements?
1. What else have you tried that worked or didn't?

**Student's answer:**

We used k-mer counting to turn DNA strings into numbers. A k-mer is just a short substring of length k. For example, for k=2 the sequence "ACGT" contains "AC", "CG", "GT". We counted all possible substrings of length 1, 2, 3, and 4, which gave us 340 features per sequence.

We chose this approach because DNA is not regular text, so language models like SBERT don't work well on it. K-mer counts are a common method in bioinformatics and they capture useful patterns in the sequence.
Since sequences have different lengths, we divided each count by the total length to get frequencies instead of raw counts. Then we applied StandardScaler so all features have the same scale, which is important for SVM to work properly.

To check if our features are good, we used 5-fold cross-validation. The accuracy was about 0.983, which shows the model generalizes well and doesn't overfit.

## **Task 2. Modeling Approach**
Explain your modeling approach, i.e. ideas you tried and why you thought they would be helpful.

1. How did these decisions guide you in modeling?
1. How do you evaluate the effectiveness of these elements?
1. What else have you tried that worked or didn't?

**Student's answer:**

We used LinearSVC with C=1.0. We picked this model because LinearSVC is fast enough to handle 100000 samples with 340 features. It worked well because our k-mer features make the two classes mostly linearly separable.

We tried several values of C (0.5, 0.8, 1.0, 1.5), but they all gave almost the same cross-validation score around 0.983. We kept 1.0 as a reasonable default. We set max_iter=20000 to make sure the model converges.

The main improvement over the baseline was using the full training set and the full DNA sequences. The baseline only used 10000 samples and 50 characters per sequence through SBERT, which gave much lower accuracy.
We considered RBF kernel SVC but it would be too slow on 100000 samples. NuSVC gave similar results to LinearSVC with no clear advantage.

[SBERT](https://www.sbert.net) generates 384-dimensional text embedding vectors for each text entry. See [more models](https://www.sbert.net/docs/pretrained_models.html).
* Only reputable publicly available embedding models are allowed (SBERT, USE, MUSE, LASER, ...). We want to prevent participants' training embeddings on test data.

In [9]:
%%capture
sbert = SBERT('paraphrase-MiniLM-L6-v2')  # 4 seconds; loads SBERT embeddings model

In [10]:
tYX = tYX.head(10000)
%time tEmb, vEmb = sbert.encode([s[:50] for s in tYX.DNA]), sbert.encode([s[:50] for s in vX.DNA])  # embed all sequences to same-size vectors
print(f'Train embedding matrix size:', tEmb.shape)
pd.DataFrame(vEmb[:3,:30], index=[x[:20]+'...' for x in vX.DNA[:3]]).style.background_gradient(cmap='coolwarm', axis=1).format("{:.2f}") # show movie description and a few of its embedding features

CPU times: user 14.7 s, sys: 249 ms, total: 15 s
Wall time: 13.3 s
Train embedding matrix size: (10000, 384)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29
TTGATTAATAAGATTCCTTG...,-0.51,-0.01,-0.09,0.00,-0.61,0.13,0.80,-0.33,0.27,-0.22,0.35,-0.40,-0.02,-0.07,-0.05,0.19,-0.03,0.82,0.10,-0.00,0.46,-0.68,-0.36,-0.08,0.50,0.33,-0.45,0.10,0.70,0.29
ATTAGTAACGGAGGATTTAC...,-0.61,0.03,-0.36,0.26,-0.10,-0.06,0.81,-0.30,0.42,-0.11,0.25,-0.40,0.09,-0.06,-0.23,-0.01,0.30,0.52,0.17,0.21,0.73,-0.39,-0.30,-0.02,0.46,-0.06,-0.80,-0.08,0.37,0.18
AAAAAGCCGTAAAAGACGAT...,-0.30,-0.03,-0.29,0.14,-0.30,0.17,1.02,-0.17,0.19,0.06,0.37,-0.87,-0.23,0.07,-0.14,-0.07,0.04,-0.07,0.17,0.36,0.22,-0.15,-0.41,0.02,0.30,0.36,-0.68,0.09,0.35,0.18


In [11]:
from itertools import product as iprod
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.svm import LinearSVC
import numpy as np, pandas as pd

tYX_full = pd.read_csv('trainYX/trainYX.csv').set_index('id')

def kmer_features(dna_series, ks=[1,2,3,4]):
    bases = 'ACGT'
    all_kmers = []
    for k in ks:
        all_kmers.extend([''.join(x) for x in iprod(bases, repeat=k)])
    features = np.zeros((len(dna_series), len(all_kmers)))
    for i, seq in enumerate(dna_series):
        for j, kmer in enumerate(all_kmers):
            features[i, j] = seq.count(kmer)
    return features, all_kmers

print('Building k-mer features (k=1,2,3,4)...')
%time train_feats, kmer_names = kmer_features(tYX_full.DNA.values, ks=[1,2,3,4])
%time test_feats, _ = kmer_features(vX.DNA.values, ks=[1,2,3,4])
print(f'Train features: {train_feats.shape}, Test features: {test_feats.shape}')

train_lens = tYX_full.DNA.str.len().values.reshape(-1,1)
test_lens = vX.DNA.str.len().values.reshape(-1,1)
train_feats_norm = train_feats / train_lens
test_feats_norm = test_feats / test_lens

scaler = StandardScaler()
X_train = scaler.fit_transform(train_feats_norm)
X_test = scaler.transform(test_feats_norm)
y_train = tYX_full.Y.values

print(f'Scaled feature matrix: {X_train.shape}')


Building k-mer features (k=1,2,3,4)...
CPU times: user 43.9 s, sys: 155 ms, total: 44 s
Wall time: 44.4 s
CPU times: user 9.07 s, sys: 40.8 ms, total: 9.11 s
Wall time: 9.81 s
Train features: (100000, 340), Test features: (20000, 340)
Scaled feature matrix: (100000, 340)


In [12]:
%time m = LinearSVC(C=1.0, random_state=0, max_iter=20000).fit(X_train, y_train)
print(f'In-sample accuracy: {m.score(X_train, y_train):.5f}')

CPU times: user 6.7 s, sys: 235 ms, total: 6.93 s
Wall time: 7.02 s
In-sample accuracy: 0.98437


In [13]:
pY = pd.DataFrame(m.predict(X_test), index=vX.index, columns=['y'])
pY.to_csv('MySubmission.csv', index_label='id')
print(f'Submission saved: {pY.shape}')
print(pY.y.value_counts())

Submission saved: (20000, 1)
y
0    10040
1     9960
Name: count, dtype: int64


In [15]:
!kaggle competitions submit -c 26-hse-genomics -f MySubmission.csv -m "first try"


100% 176k/176k [00:00<00:00, 289kB/s]
Successfully submitted to 26hse-🧬Genomics

# **References:**

1. Scikit-learn documentation: Support Vector Machines. https://scikit-learn.org/stable/modules/svm.html
2. Melsted, P. & Pritchard, J.K. (2011). Efficient counting of k-mers in DNA sequences using a bloom filter. BMC Bioinformatics, 12, 333.
3. Noble, W.S. (2006). What is a support vector machine? *Nature Biotechnology*, 24(12), 1565–1567.
4. Khan Academy. The genetic code. https://www.khanacademy.org/science/ap-biology/gene-expression-and-regulation/translation/a/the-genetic-code-discovery-and-properties
5. Medium: Apply Machine Learning Algorithms for Genomics Data Classification. https://medium.com/mlearning-ai/apply-machine-learning-algorithms-for-genomics-data-classification-132972933723

<font color=green><h4><b>$\epsilon$. LLM Documentation if used</b></h4></font>

<font color=red><b>Your answer here.</b></font>

<font size=5>⌛</font> <strong><font color=orange size=5>Do not exceed competition's runtime limit!</font></strong>

<hr color=red>


In [14]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.

Runtime is 110 sec


# 💡**Starter Ideas**

1. Learn about DNA [&#127910;](https://www.youtube.com/results?search_query=nucleotides+genes+amino+acids+)
1. Try a larger training sample.
1. Try longer training DNA strings, but SBERT may have a cap on string length, so you might split DNA into several strings and then concatenate or average resulting vectors
1. Try other pretrained SBERT models. Note that DNA sequence uses ACGT letters, but many other models were trained on multilingual text. So, you might prefer those that were trained on mostly ASCII.
1. SBERT is trained on word tokens (typically, separated by spaces), but DNA sequence has no spaces. Try placing spaces after every character or some semantically meaningful subsequences (this might require more domain knowledge).
1. Try Google's [USE](https://tfhub.dev/google/universal-sentence-encoder-multilingual/3) embedding models
1. Try Facebook's [LASER](https://github.com/facebookresearch/LASER) and [others](https://tfhub.dev/s?module-type=text-language-model).
1. Try [Enformer](https://tfhub.dev/deepmind/enformer/1) for gene expressions. See [DeepMind paper](https://deepmind.com/blog/article/enformer).
1. Try building your own embeddings on the given sequences. SBERT and other packages make it easy (just a few lines), but it may take too much time.
1. Assess distribution of character patterns (single, doubles, triplets, ...). For example, an ACGT string generates AC, CG, GT doubles and ACG and CGT triplets. Does one class have more subsequences of some type? This might be a feature in your model.
1. Try features built as counts of subsequences (singles, doubles, triplets, ...). Consider EDA first.
1. Concatenate or otherwise combine multiple embeddings derived from each gene string
1. Learn from [*The genetic code*](https://www.khanacademy.org/science/ap-biology/gene-expression-and-regulation/translation/a/the-genetic-code-discovery-and-properties), Khan Academy.
1. Learn from [*Apply Machine Learning Algorithms for Genomics Data Classification*](https://medium.com/mlearning-ai/apply-machine-learning-algorithms-for-genomics-data-classification-132972933723)
1. Learn from [*Efficient counting of k-mers in DNA sequences using a bloom filter*](https://bmcbioinformatics.biomedcentral.com/articles/10.1186/1471-2105-12-333) Páll Melsted et al. 2011
1. Try [Byte Pair Encoding](https://www.derczynski.com/papers/archive/BPE_Gage.pdf) and [SentencePiece](https://arxiv.org/pdf/1808.06226.pdf) to auto identification of "important" [k-mers](https://en.wikipedia.org/wiki/K-mer) (substrings)